In [ ]:
# Written by tools/build_notebooks.py -- do not edit. The bootstrap cell below
# compares this against the repo it clones and tells you if these cells are old.
CELLS_SRC = "kaggle_01_audit_and_tier_a.py"
CELLS_SHA = "be567dd1d67e4979"

# 01 — The audit, and the counting half on CPU

**Accelerator: None (CPU).** No GPU quota. **Runtime: about 25–40 minutes.**

## Read this bit

**This notebook is the deliverable.** Not a warm-up, not a baseline to beat later — if you
run only one notebook in this project, run this one. When it finishes you have every number
the course brief asks for, and you have spent zero of your 30 GPU-hours.

The reason is a measurement. In the sibling project, a 5.3-million-parameter neural network
scored **20.00 %** at counting speakers — exactly chance on a 5-class problem. Sixteen
hand-crafted numbers fed to a gradient-boosted tree scored **69.3 %** on the same data. A
decision tree beat the network by 49 points. `docs/DIAGNOSIS.md` explains why.

So we do the cheap, certain thing first and properly, and treat the neural network
(notebook 02) as optional upside.

## What runs here

| step | what it answers |
|---|---|
| **the audit** | Is the evaluation honest? Can a model cheat by reading loudness instead of listening? |
| **naive baselines** | What do you get for free, with no learning at all? |
| **Tier A** | Feature extraction, a speaker-disjoint K-fold search, the final model |
| **feature importance** | *Why* does it know? Every answer is a named acoustic quantity |

## Before you press Run

1. **Settings → Accelerator → None**
2. **+ Add Input → Notebook Output →** the output of notebook 00
3. **Settings → Internet → On** (only if the code comes from GitHub)

## Bootstrap (this cell is identical in every notebook)

Three ways to get the code onto the Kaggle machine, tried in order:

1. **GitHub clone** — set `REPO_URL` below and turn *Internet* ON in the notebook
   settings panel (Settings → Internet → On). This is the recommended route.
2. **Repo-as-dataset** — upload this folder as a Kaggle Dataset called
   `speaker-count-separate-v1` and attach it. No internet needed. Use this if your
   account cannot enable internet (phone-verification is required for that).
3. **Already there** — an existing clone is **fast-forwarded to the newest commit**,
   not reused as-is. A Kaggle session outlives many pushes, and silently running code
   from an hour ago is the most expensive kind of confusion: the log looks fine and the
   fix you are testing is not in it. Any local edits inside the clone are discarded.

Whichever route runs, the commit is printed. Every log can then be traced to the exact
code that produced it.

In [ ]:
REPO_URL = "https://github.com/AlAminAshraf01/speaker-count-separate-v1.git"
REPO_DIR = "/kaggle/working/speaker-count-separate-v1"
REPO_AS_DATASET = "/kaggle/input/speaker-count-separate-v1"

import hashlib
import os
import shutil
import subprocess
import sys


def cells_fingerprint(src_dir: str, name: str) -> str:
    """Short hash of one notebook's percent source plus this shared bootstrap.

    ``tools/build_notebooks.py`` stamps this into every generated ``.ipynb``. The copy
    running on Kaggle recomputes it from the freshly-cloned repo, so a notebook whose
    cells were imported before the last push says so in the first ten seconds instead of
    eleven hours later.

    Line endings are normalised first. The same file is CRLF in a Windows working tree
    and LF in a Linux clone, and a fingerprint that disagrees with itself across
    platforms is worse than no fingerprint at all.
    """
    digest = hashlib.sha256()
    for part in (name, "_bootstrap.py"):
        with open(os.path.join(src_dir, part), "rb") as fh:
            digest.update(fh.read().replace(b"\r\n", b"\n"))
        digest.update(b"\0")
    return digest.hexdigest()[:16]


def cells_status(repo_dir: str, src_name: str | None, stamp: str | None) -> str:
    """Compare the stamp baked into these cells with the repo they are about to run.

    Never raises. A check that can take down every notebook is a worse bug than the one
    it detects, so anything unreadable degrades to "cannot verify".
    """
    if not src_name or not stamp:
        return "unstamped -- re-import this notebook to enable the staleness check"
    try:
        current = cells_fingerprint(os.path.join(repo_dir, "notebooks", "src"), src_name)
    except Exception as exc:
        return f"cannot verify ({exc})"
    if current == stamp:
        return f"current ({stamp})"
    return "\n".join([
        f"STALE  cells {stamp} but repo has {current}",
        "",
        "  These notebook cells were imported before the newest push, so the fix you",
        "  are about to test is not in them. scripts/ and src/ just updated themselves;",
        "  notebook cells cannot, because Kaggle owns them.",
        "",
        "  Fix: File -> Import Notebook -> upload notebooks/" + src_name[:-3] + ".ipynb",
        "       again, re-attach the inputs, and re-run.",
    ])


def _git(repo_dir: str, *argv: str) -> subprocess.CompletedProcess:
    return subprocess.run(["git", "-C", repo_dir, *argv],
                          capture_output=True, text=True)


def update_clone(repo_dir: str) -> str:
    """Fast-forward an existing clone to the remote's newest commit.

    Returns a short status for printing; never raises. Losing internet is a reason to
    carry on with the code that is already there, but it is not a reason to be quiet
    about it -- running stale code unknowingly is how a fix gets tested without being
    present.
    """
    if not os.path.isdir(os.path.join(repo_dir, ".git")):
        return "not a git clone, left as it is"
    branch = _git(repo_dir, "rev-parse", "--abbrev-ref", "HEAD").stdout.strip() or "main"
    before = _git(repo_dir, "rev-parse", "--short", "HEAD").stdout.strip()
    fetched = _git(repo_dir, "fetch", "--depth", "1", "origin", branch)
    if fetched.returncode != 0:
        tail = (fetched.stderr or "").strip().splitlines()
        return f"COULD NOT FETCH ({tail[-1] if tail else 'unknown'}) -- code may be stale"
    reset = _git(repo_dir, "reset", "--hard", f"origin/{branch}")
    if reset.returncode != 0:
        tail = (reset.stderr or "").strip().splitlines()
        return f"COULD NOT UPDATE ({tail[-1] if tail else 'unknown'}) -- code may be stale"
    after = _git(repo_dir, "rev-parse", "--short", "HEAD").stdout.strip()
    return "already newest" if before == after else f"updated {before} -> {after}"


def describe_commit(repo_dir: str) -> str:
    """``<short sha> <date> <subject>`` for the checked-out commit, or a plain note."""
    out = _git(repo_dir, "log", "-1", "--format=%h %cs %s").stdout.strip()
    return out or "no git metadata"


def bootstrap(repo_url: str = REPO_URL, repo_dir: str = REPO_DIR) -> str:
    """Put the repo at `repo_dir`, put its `src/` on sys.path, and chdir into it."""
    if not os.path.isdir(os.path.join(repo_dir, "src")):
        if os.path.isdir(os.path.join(REPO_AS_DATASET, "src")):
            shutil.copytree(REPO_AS_DATASET, repo_dir, dirs_exist_ok=True)
            print(f"copied repo from the attached dataset {REPO_AS_DATASET}")
        else:
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_dir], check=True)
            print(f"cloned {repo_url}")
    else:
        print(f"existing clone: {update_clone(repo_dir)}")
    src = os.path.join(repo_dir, "src")
    if src not in sys.path:
        sys.path.insert(0, src)
    os.chdir(repo_dir)
    return repo_dir


REPO = bootstrap()

import countsep  # noqa: E402

print("countsep", countsep.__version__, "at", REPO)
print("code ", describe_commit(REPO))
# CELLS_SRC / CELLS_SHA are set by the stamp cell that tools/build_notebooks.py puts at
# the top of every generated notebook. globals().get keeps this working in a notebook
# assembled by hand, where that cell may not exist.
CELLS = cells_status(REPO, globals().get("CELLS_SRC"), globals().get("CELLS_SHA"))
print("cells", CELLS if "\n" not in CELLS else "")
if "\n" in CELLS:
    print(CELLS)
print("python", sys.version.split()[0])

import torch  # noqa: E402

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| devices", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  [{i}] {p.name}  {p.total_memory / 1e9:.1f} GB")

In [ ]:
import shlex
import time


def run(cmd: str, check: bool = True) -> int:
    """Run a shell command, streaming its output into the notebook."""
    print("$", cmd, flush=True)
    t0 = time.time()
    proc = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    print(f"\n[exit {code} in {time.time() - t0:.1f}s]", flush=True)
    if check and code != 0:
        raise SystemExit(f"command failed with exit code {code}")
    return code

In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO, "scripts"))
from _common import autodetect_store, find_recipes

STORE = autodetect_store()
RECIPES_DEV = find_recipes("recipes_dev.csv", STORE)
RECIPES_TEST = find_recipes("recipes_test.csv", STORE)
OUT = "/kaggle/working/tier_a"

print("store       :", STORE)
print("recipes dev :", RECIPES_DEV)
print("recipes test:", RECIPES_TEST)
if STORE is None:
    raise SystemExit(
        "No packed store found. In the right-hand panel: '+ Add Input' -> 'Notebook Output'\n"
        "-> pick your run of notebook 00. Then re-run this cell.")

run(f"python scripts/preflight.py --for cpu --store {STORE}"
    f" --cells_src {CELLS_SRC} --cells_sha {CELLS_SHA}")

## Step 1 — the leakage audit  (~5 min)

### What "leakage" means here, in plain words

When you build a mixture by adding up N voices, the result gets **louder** as N grows. If you
leave it that way, a model can score well by measuring loudness and never listening to
speech at all. That is not a speaker counter; it is a volume meter, and it will collapse the
moment it meets real audio.

The fix is structural: every clip is cut to exactly 3 seconds and divided by its own loudness.
After that, loudness and length are **identical for every clip at every N**, so there is
nothing left to cheat on.

### How this script proves it rather than claiming it

It runs the same probe twice:

* on audio **before** the fix — this *must* score well above chance, or the probe is broken
  and its clean verdict would be worthless;
* on audio **after** the fix — this must sit at chance.

> ### Why this script is built to fail loudly
>
> The equivalent script in the sibling project was **dead code for months**. Somebody pasted
> a function into the middle of `main()`, which quietly cut the rest of it off; the script
> printed its heading, wrote nothing, and exited with code 0 — *success*. Two headline numbers
> were quoted from it in project documents that it had never actually produced.
>
> So this one checks that its own report file exists before it exits, and the test suite
> fails the build on that exact bug shape.

In [ ]:
run(f"python scripts/02_audit_and_baselines.py"
    f" --store {STORE}"
    f" --splits train-100 dev test"
    f" --probe_split train-100"
    f" --per_class 300"
    f" --out /kaggle/working/audit")

## Step 2 — Tier A: the model  (~20–30 min)

Sixteen acoustic numbers per clip, then a gradient-boosted tree.

### The idea in one sentence

When you add up more and more voices, the mixture **fills in its own silences** and starts to
look like plain noise — so statistics that measure "peakiness" fall in a predictable way as
the number of talkers goes up. Measured here: kurtosis spreads **5×** across 1 to 5 speakers.

### The K-fold is grouped by speaker, and that matters

An ordinary K-fold would put the same person in both the training and the validation half.
LibriSpeech speakers can be identified from their vocabulary alone, so the tree would be
partly recognising *voices* rather than counting them, and the score would be flattering and
wrong. Here the folds are cut so **no speaker crosses the line**. The number this prints is
usually a few points lower than the naive version. It is the honest one.

In [ ]:
run(f"python scripts/03_tier_a.py"
    f" --store {STORE}"
    f" --train_split train-100"
    f" --per_class 1500"
    f" --folds 5"
    f" --out {OUT}")

## Step 3 — read the result

The last table of the previous cell is your **interpretability deliverable**: it ranks the
acoustic quantities by how much the model actually relies on them, and every row has a name
you can say out loud — "envelope sparsity", "kurtosis" — instead of "filter 287".

One honest caveat to write in your report: permutation importance **splits the credit between
correlated features**. Kurtosis of the signal and kurtosis of the envelope measure nearly the
same thing, so one of them can show near-zero importance while the pair is jointly essential.
Do not read a zero as "this feature is useless".

In [ ]:
import json

with open(os.path.join(OUT, "tier_a_report.json")) as fh:
    report = json.load(fh)

bar = report["bar"]
print(f"chance                                 20.0%")
print(f"best naive predictor                   "
      f"{max(r['accuracy'] for r in report['naive'].values()):.1%}")
print(f"Tier A, speaker-disjoint K-fold        {bar:.1%}   <-- YOUR RESULT")
print()
print(f"For comparison, the 5.30 M joint model in the sibling repo scored 20.00 %.")
print()
print(f"Pass this to notebook 02 as --bar {bar:.3f}")

## You are done — this is a complete project

**Save Version → Save & Run All (Commit)** so the model and the reports become a dataset.

What you have now, mapped to the five things the course asks for:

| phase | where it is |
|---|---|
| 1 · EDA, correlations, outliers | feature distributions per N + the importance table |
| 2 · custom feature extraction | the 16 acoustic features — a hand-designed transform |
| 3 · hyperparameter search + K-fold | the grouped 5-fold search in step 2 |
| 4 · data-leakage audit | step 1, which actually runs and proves its own probe works |
| 5 · baselines + interpretability | three naive predictors + permutation importance |

notebooks 02 and 03 (the neural models) is **optional**. If your GPU quota is gone, or the week is
gone, stop here and write it up. That is not settling — that is the plan working.